# NB3 — Extensión: Hard Negative Mining y RoBERTa

**Plataforma:** Vast.ai (A100) — requiere GPU.

## Propósito
Implementar y evaluar dos extensiones al método AAN del paper original:

### Extensión 1: Hard Negative Mining
El paper original muestrea negativos **aleatoriamente** del batch. El problema
es que muchos negativos fáciles (representaciones ya alejadas) contribuyen
señal de gradiente casi nula — el término (1 + cos_sim) ya está cerca de 0.

La mejora propuesta selecciona los n negativos con **mayor similitud coseno**
al ancla — los más difíciles de distinguir para el modelo actual. Esto produce
señal de gradiente más fuerte y fuerza al encoder a trabajar en los casos
donde actualmente falla más.

El cambio es mínimo: una línea en la selección de negativos:
```python
# AAN original (aleatorio):
neg_idx = diff_indices[torch.randperm(...)[:n]]

# AAN hard (por similitud):
neg_idx = diff_indices[torch.topk(cos_sims, n).indices]
```

### Extensión 2: Encoder más fuerte (RoBERTa-base)
RoBERTa-base tiene la misma arquitectura que BERT-base (~125M parámetros)
pero fue entrenado con mayor cantidad de datos, sin la tarea de Next Sentence
Prediction, y con mejores hiperparámetros. Esto lo convierte en un candidato
natural para evaluar si los beneficios de AAN escalan con la calidad del encoder.

## Modelos entrenados
1. BERT + AAN hard negatives
2. RoBERTa baseline
3. RoBERTa + AAN random negatives
4. RoBERTa + AAN hard negatives

Los resultados de BERT baseline y BERT AAN random vienen de NB2.

## Prerequisito
NB1 y NB2 deben haberse ejecutado. `nb2_results.json` debe existir.

## 1. Rutas e importaciones

In [ ]:
import os, sys
os.environ['HF_HOME'] = '/workspace/hf_cache'

from pathlib import Path
BASE_DIR = Path('/workspace/negative_supervision')
DATA_DIR = BASE_DIR / 'data'
CKPT_DIR = BASE_DIR / 'checkpoints'
RES_DIR  = BASE_DIR / 'results'
for d in [CKPT_DIR, RES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / 'label_info.json').exists(), 'Ejecutar NB1 primero.'
assert (RES_DIR / 'nb2_results.json').exists(),  'Ejecutar NB2 primero.'
print(f'Base: {BASE_DIR}')

In [ ]:
!{sys.executable} -m pip install -q transformers==4.40.0 datasets==2.19.0 scikit-learn==1.4.2 pandas numpy sentencepiece protobuf

In [ ]:
import json, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

with open(DATA_DIR / 'label_info.json') as f:
    LABEL_INFO = json.load(f)
with open(RES_DIR / 'nb2_results.json') as f:
    NB2_RESULTS = json.load(f)

print(f'Resultados NB2 disponibles para: {list(NB2_RESULTS.keys())}')
print('Importaciones OK.')

## 2. Clases de Dataset (idénticas a NB2)

In [ ]:
class SingleLabelDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts, self.labels = df['text'].tolist(), df['label'].tolist()
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

class MultiLabelDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['text'].tolist()
        self.label_vecs = df['label_vec'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x).tolist()
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.label_vecs[idx], dtype=torch.float)}

## 3. Definición de modelos

Se definen tres modelos. `BaselineClassifier` y `AANClassifier` son idénticos a NB2
y se reutilizan con RoBERTa pasando `encoder_name='roberta-base'`.
`AANHardClassifier` es la contribución nueva.

In [ ]:
class BaselineClassifier(nn.Module):
    """Encoder + cabeza lineal. Full fine-tuning estándar."""
    def __init__(self, encoder_name, num_labels, task_type):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(encoder_name)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.task_type  = task_type
    def encode(self, input_ids, attention_mask):
        return self.encoder(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state[:, 0, :]
    def forward(self, input_ids, attention_mask, labels=None):
        logits = self.classifier(self.encode(input_ids, attention_mask))
        loss = None
        if labels is not None:
            loss = (F.cross_entropy(logits, labels) if self.task_type == 'single_label'
                    else F.binary_cross_entropy_with_logits(logits, labels))
        return loss, logits


class AANClassifier(nn.Module):
    """AAN con muestreo ALEATORIO de negativos — igual que en NB2."""
    def __init__(self, encoder_name, num_labels, task_type, n_negatives=4):
        super().__init__()
        self.encoder     = AutoModel.from_pretrained(encoder_name)
        self.classifier  = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.task_type, self.n_negatives = task_type, n_negatives
    def encode(self, input_ids, attention_mask):
        return self.encoder(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state[:, 0, :]
    def auxiliary_loss(self, v, labels):
        total_loss, count, v_pool = torch.tensor(0.0, device=v.device), 0, v.detach()
        for i in range(v.size(0)):
            diff_mask = (labels != labels[i]) if self.task_type == 'single_label' \
                        else ~(labels == labels[i].unsqueeze(0)).all(dim=1)
            diff_idx = diff_mask.nonzero(as_tuple=True)[0]
            if len(diff_idx) == 0: continue
            n = min(self.n_negatives, len(diff_idx))
            neg_idx = diff_idx[torch.randperm(len(diff_idx), device=v.device)[:n]]
            cos_sim = F.cosine_similarity(v[i].unsqueeze(0).expand(n,-1), v_pool[neg_idx], dim=1)
            total_loss = total_loss + (1.0 + cos_sim).mean()
            count += 1
        return total_loss / count if count > 0 else torch.tensor(0.0, device=v.device, requires_grad=True)
    def forward(self, input_ids, attention_mask, labels=None):
        v = self.encode(input_ids, attention_mask)
        logits = self.classifier(v)
        loss = None
        if labels is not None:
            lm = (F.cross_entropy(logits, labels) if self.task_type == 'single_label'
                  else F.binary_cross_entropy_with_logits(logits, labels))
            loss = lm + self.auxiliary_loss(v, labels)
        return loss, logits


class AANHardClassifier(nn.Module):
    """
    AAN con HARD NEGATIVE MINING — contribución de este trabajo.

    Diferencia respecto a AANClassifier:
    En lugar de muestrear n negativos aleatoriamente, se computa la similitud
    coseno entre el ancla y TODOS los negativos válidos del batch, y se
    seleccionan los n con MAYOR similitud (los más difíciles).

    Esto concentra la señal de gradiente en los casos donde el encoder
    actualmente falla más, produciendo una supervisión más eficiente.

    Cambio clave (una línea):
        Aleatorio: neg_idx = diff_indices[torch.randperm(...)[:n]]
        Hard:      neg_idx = diff_indices[torch.topk(cos_sims, n).indices]

    La función de pérdida auxiliar es idéntica al AAN original:
        La = (1/n) * Σ_j [1 + cosine_similarity(v_anchor, v_neg_j)]
    """
    def __init__(self, encoder_name, num_labels, task_type, n_negatives=4):
        super().__init__()
        self.encoder     = AutoModel.from_pretrained(encoder_name)
        self.classifier  = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.task_type, self.n_negatives = task_type, n_negatives

    def encode(self, input_ids, attention_mask):
        return self.encoder(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state[:, 0, :]

    def auxiliary_loss(self, v, labels):
        total_loss, count, v_pool = torch.tensor(0.0, device=v.device), 0, v.detach()
        for i in range(v.size(0)):
            diff_mask = (labels != labels[i]) if self.task_type == 'single_label' \
                        else ~(labels == labels[i].unsqueeze(0)).all(dim=1)
            diff_idx = diff_mask.nonzero(as_tuple=True)[0]
            if len(diff_idx) == 0: continue

            # Computar similitud con TODOS los negativos del batch
            v_anc_exp = v[i].unsqueeze(0).expand(len(diff_idx), -1)
            all_sims  = F.cosine_similarity(v_anc_exp, v_pool[diff_idx], dim=1)

            # Seleccionar los n MÁS DIFÍCILES (mayor similitud coseno)
            n       = min(self.n_negatives, len(diff_idx))
            neg_idx = diff_idx[torch.topk(all_sims, n).indices]

            cos_sim    = F.cosine_similarity(v[i].unsqueeze(0).expand(n,-1), v_pool[neg_idx], dim=1)
            total_loss = total_loss + (1.0 + cos_sim).mean()
            count     += 1

        return total_loss / count if count > 0 else torch.tensor(0.0, device=v.device, requires_grad=True)

    def forward(self, input_ids, attention_mask, labels=None):
        v = self.encode(input_ids, attention_mask)
        logits = self.classifier(v)
        loss = None
        if labels is not None:
            lm = (F.cross_entropy(logits, labels) if self.task_type == 'single_label'
                  else F.binary_cross_entropy_with_logits(logits, labels))
            loss = lm + self.auxiliary_loss(v, labels)
        return loss, logits

## 4. Métricas, entrenamiento y utilidades (idénticas a NB2)

In [ ]:
def accuracy(logits, labels):
    return (logits.argmax(dim=1) == labels).float().mean().item()

def exact_match(logits, labels, threshold=0.5):
    return ((torch.sigmoid(logits) >= threshold).float() == labels).all(dim=1).float().mean().item()

def compute_metric(logits, labels, task_type):
    return accuracy(logits, labels) if task_type == 'single_label' else exact_match(logits, labels)

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        loss, _ = model(batch['input_ids'].to(DEVICE),
                        batch['attention_mask'].to(DEVICE),
                        batch['label'].to(DEVICE))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, task_type):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        _, logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        all_logits.append(logits); all_labels.append(batch['label'].to(DEVICE))
    return compute_metric(torch.cat(all_logits), torch.cat(all_labels), task_type)

def run_single_trial(model_class, model_kwargs, train_loader, val_loader,
                     test_loader, task_type, lr, seed, patience=10, max_epochs=50):
    set_seed(seed)
    model = model_class(**model_kwargs).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.999, 0.9))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(0.1*max_epochs*len(train_loader)), max_epochs*len(train_loader))
    best_val, best_state, no_improve = -1.0, None, 0
    for epoch in range(1, max_epochs+1):
        train_epoch(model, train_loader, optimizer, scheduler)
        val_m = evaluate(model, val_loader, task_type)
        if val_m > best_val:
            best_val, best_state, no_improve = val_m, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f'      Early stop época {epoch} | val={val_m:.4f}'); break
    model.load_state_dict(best_state)
    return evaluate(model, test_loader, task_type), best_val, model

def run_experiment(model_class, model_kwargs, train_loader, val_loader,
                   test_loader, task_type, lr, n_trials=5, ckpt_path=None):
    test_scores, best_val_overall, best_state = [], -1.0, None
    for t in range(n_trials):
        print(f'    Trial {t+1}/{n_trials} ...')
        test_m, val_m, model = run_single_trial(
            model_class, model_kwargs, train_loader, val_loader,
            test_loader, task_type, lr, seed=t)
        test_scores.append(test_m)
        print(f'      val={val_m:.4f} | test={test_m:.4f}')
        if val_m > best_val_overall:
            best_val_overall, best_state = val_m, copy.deepcopy(model.state_dict())
    trimmed = sorted(test_scores)[1:-1]
    mean_score, std_score = float(np.mean(trimmed)), float(np.std(trimmed))
    if ckpt_path:
        ckpt_path.mkdir(parents=True, exist_ok=True)
        torch.save(best_state, ckpt_path / 'best_model.pt')
        print(f'    Checkpoint → {ckpt_path}')
    print(f'    => Media trimmed: {mean_score:.4f} ± {std_score:.4f}')
    return {'mean': round(mean_score,4), 'std': round(std_score,4),
            'all_test_scores': test_scores, 'trimmed_scores': trimmed}

def load_splits(dataset_name, tokenizer, batch_size=16):
    info = LABEL_INFO[dataset_name]
    path = DATA_DIR / dataset_name
    Cls  = SingleLabelDataset if info['task_type'] == 'single_label' else MultiLabelDataset
    return (
        DataLoader(Cls(pd.read_csv(path/'train.csv'), tokenizer),
                   batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True),
        DataLoader(Cls(pd.read_csv(path/'val.csv'),   tokenizer),
                   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True),
        DataLoader(Cls(pd.read_csv(path/'test.csv'),  tokenizer),
                   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True),
        info['task_type'], info['num_labels']
    )

def select_lr(model_class, model_kwargs, train_loader, val_loader,
              test_loader, task_type, candidates=[1e-5, 3e-5, 5e-5]):
    best_lr, best_val = None, -1.0
    for lr in candidates:
        print(f'    LR={lr} ...')
        _, val_m, _ = run_single_trial(model_class, model_kwargs, train_loader,
                                       val_loader, test_loader, task_type, lr, seed=99)
        print(f'      val={val_m:.4f}')
        if val_m > best_val: best_val, best_lr = val_m, lr
    print(f'    Mejor LR: {best_lr}')
    return best_lr

## 5. Ejecutar experimentos

4 configuraciones × 2 datasets × (3 LR + 5 trials) = 32 runs.
Guardado incremental después de cada experimento.

In [ ]:
BERT_NAME    = 'bert-base-uncased'
ROBERTA_NAME = 'roberta-base'
NB3_RESULTS  = {}

EXPERIMENTS = [
    ('bert_aan_hard',    BERT_NAME,    AANHardClassifier,  {'n_negatives': 4}),
    ('roberta_baseline', ROBERTA_NAME, BaselineClassifier, {}),
    ('roberta_aan',      ROBERTA_NAME, AANClassifier,      {'n_negatives': 4}),
    ('roberta_aan_hard', ROBERTA_NAME, AANHardClassifier,  {'n_negatives': 4}),
]

for dataset_name in ['mr', 'semeval']:
    print(f'\n{"="*60}')
    print(f'DATASET: {dataset_name.upper()}')
    print(f'{"="*60}')
    NB3_RESULTS[dataset_name] = {}

    for exp_key, encoder_name, model_class, extra in EXPERIMENTS:
        print(f'\n  --- {exp_key.upper()} ---')
        tokenizer = AutoTokenizer.from_pretrained(encoder_name)
        train_loader, val_loader, test_loader, task_type, num_labels = \
            load_splits(dataset_name, tokenizer)

        model_kwargs = {'encoder_name': encoder_name,
                        'num_labels': num_labels, 'task_type': task_type, **extra}

        print('  Seleccionando LR...')
        best_lr = select_lr(model_class, model_kwargs, train_loader,
                            val_loader, test_loader, task_type)

        print(f'  Ejecutando 5 trials con LR={best_lr}...')
        result = run_experiment(model_class, model_kwargs, train_loader,
                                val_loader, test_loader, task_type, best_lr,
                                n_trials=5,
                                ckpt_path=CKPT_DIR/f'{exp_key}_{dataset_name}')
        result['best_lr'] = best_lr
        NB3_RESULTS[dataset_name][exp_key] = result

        with open(RES_DIR / 'nb3_results.json', 'w') as f:
            json.dump(NB3_RESULTS, f, indent=2)
        print('  Guardado.')

## 6. Resumen

In [ ]:
with open(RES_DIR / 'nb3_results.json') as f:
    NB3_RESULTS = json.load(f)

print('Resultados NB3 — Hard Negatives + RoBERTa')
for ds in ['mr', 'semeval']:
    print(f'\n{ds.upper()}:')
    print(f'  {"Modelo":<25} {"Media":>8} {"Std":>8}')
    print('  ' + '-'*43)
    for exp_key in ['bert_aan_hard','roberta_baseline','roberta_aan','roberta_aan_hard']:
        if exp_key in NB3_RESULTS.get(ds, {}):
            r = NB3_RESULTS[ds][exp_key]
            print(f'  {exp_key:<25} {r["mean"]:>8.4f} {r["std"]:>8.4f}')
print()
print('NB3 completo. Continuar con NB4.')